# 大模型基础流程

## 环境准备

本notebook用于从最基本的大模型的基本步骤入手，展示大模型生成下一个词过程中的基本概念与基本流程，主要目的是走通最基本的流程，通过代码展示每一步骤实际所做的事情。首先导入相关需要的库，其中的transformers即是Hugging Face开源的深度学习库，提供了大量预训练Transformer模型的统一接口，可以理解为已经搭建好的积木库。

In [1]:
import sys
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
from loguru import logger

logger.remove()
logger.add(sys.stdout, level="INFO", colorize=True);

配置代理，其中的端口号可以根据VPN的具体配置来查询，方便在中国大陆访问Hugging Face。

In [2]:
import os

os.environ["HTTP_PROXY"] = "http://127.0.0.1:6382"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:6382"

加载GPT-2的tokenizer和model以及LM Head，下面的model已经包含了模型和LM Head，其中的model.eval()指的是使用推理模式，而非训练模式。

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

### GPT2LMHeadModel 结构

GPT2LMHeadModel 由两部分组成：

- `transformer: GPT2Model`：负责将 token 转换为上下文化的 hidden representation。
- `lm_head`：负责将最终 hidden representation 映射到 vocabulary logits。

#### GPT2Model

- `wte (word token embedding)`：
  GPT-2 词表大小为 50257，每个 token ID 对应一个可训练的 768 维向量。
  因此 WTE 是一个 `[50257, 768]` 的 embedding table，实现
  `token id → 768维 hidden representation`。
  768 是 GPT-2 small 的 hidden size；模型有 12 个 attention heads，
  因此每个 head 的维度为 64。

- `wpe (word position embedding)`：
  GPT-2 最大支持 1024 个位置，因此维护 `[1024, 768]` 的位置 embedding。
  根据 position ID 查表，并与 token embedding 相加：

  hidden_state_0 = token_embedding + position_embedding

- `drop`：
  embedding 后的 Dropout，训练时 p=0.1，用于正则化；
  `model.eval()` 后推理阶段关闭。

- `h`：
  包含 12 个结构相同、参数彼此独立的 GPT2Block。

  每个 GPT2Block 包含：

  - `ln_1`：
    Attention 子层之前的 LayerNorm，对每个 token 的 768 维 feature 做归一化。

  - `attn`：
    Multi-Head Self-Attention。

    - `c_attn`：
      768 → 2304 = 3 × 768，一次性产生 Q、K、V。
      每个 Q/K/V 再划分为 12 个 head，每个 head 64 维。

    - `c_proj`：
      多个 attention head 的结果合并为 768 维之后，
      再做 768 → 768 的 output projection。

    - `attn_dropout / resid_dropout`：
      Attention 中不同位置使用的 Dropout，推理时关闭。

    注意：GPT-2 中的 Conv1D 并不是通常 CNN 中的一维卷积，
    本质上是类似 Linear 的仿射线性变换。

  - `ln_2`：
    MLP 子层之前的 LayerNorm。

  - `mlp`：
    对每个 token 的 feature 独立进行非线性变换：

    768 → 3072 → GELU → 768

    - `c_fc`：768 → 3072
    - `act`：NewGELU
    - `c_proj`：3072 → 768
    - `dropout`：训练时使用，推理时关闭

    Attention 主要混合不同 token 之间的信息；
    MLP 主要对单个 token 内部的 feature 做非线性变换。

- `ln_f`：
  12 个 GPT2Block 全部结束之后，对最终 hidden state 再做一次 LayerNorm。

#### LM Head

- `lm_head`：
  将每个位置的 768 维 hidden state 通过线性变换映射到
  50257 维 vocabulary logits：

  [B, L, 768] → [B, L, 50257]

  每个 logit 表示对应 vocabulary token 的未归一化预测分数。

  GPT-2 中 lm_head 与 wte 共享权重（weight tying）。

## 从提示词文字->token id

通过输入简单的prompt来查看其对应的token id，并且该过程是可以可逆进行查询的。

In [4]:
prompt = "Thank you very"
logger.info(f"input prompt: {prompt}")

input_ids = tokenizer.encode(prompt, return_tensors="pt")
tokens = [tokenizer.decode(input_id) for input_id in input_ids[0]]
logger.info(f"Token IDs: {input_ids.tolist()[0]}")
logger.info(f"Tokens: {tokens}")
logger.info(f"Total token number: {len(tokens)}")


2026-08-03 12:51:09.399 | INFO     | __main__:<module>:2 - input prompt: Thank you very
2026-08-03 12:51:09.400 | INFO     | __main__:<module>:6 - Token IDs: [10449, 345, 845]
2026-08-03 12:51:09.400 | INFO     | __main__:<module>:7 - Tokens: ['Thank', ' you', ' very']
2026-08-03 12:51:09.401 | INFO     | __main__:<module>:8 - Total token number: 3


## 让模型预测下一个词

使用torch.no_grad()与之前的drop类似，推理阶段不需要保存传播过程中的梯度。output.logits的维度会是[1, 3, 50257]，其中1是batch size，3是token数，50257是词表大小。对应的next_token_logits的则是包含了最后一个token

In [5]:
with torch.no_grad():
    outputs = model(input_ids)
    next_token_logits = outputs.logits[0, -1, :]
    logger.info(f"logits shape: {outputs.logits.shape}")
    logger.info(f"next_token_logits shape: {next_token_logits.shape}")

2026-08-03 12:51:09.449 | INFO     | __main__:<module>:4 - logits shape: torch.Size([1, 3, 50257])
2026-08-03 12:51:09.450 | INFO     | __main__:<module>:5 - next_token_logits shape: torch.Size([50257])


output的数据结构：

In [6]:
logger.info(outputs.__dict__)

2026-08-03 12:51:09.465 | INFO     | __main__:<module>:1 - {'loss': None, 'logits': tensor([[[-30.4689, -31.1036, -34.8649,  ..., -39.2315, -38.8750, -31.6172],
         [-65.6156, -68.0715, -72.7663,  ..., -79.4569, -80.7184, -69.5630],
         [-59.0983, -58.9518, -67.3273,  ..., -68.9031, -69.3630, -63.3151]]]), 'past_key_values': DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer]), 'hidden_states': None, 'attentions': None, 'cross_attentions': None}


### 关于output数据结构的说明：
+ loss : 训练的loss，由于现在是推理阶段，因此没有loss,如果是训练阶段可以调用的时候类似于outputs = model(input_ids=input_ids, labels=input_ids)，即可计算出loss的值。
+ logits : 原始分数，没有对齐取值范围的规定，即可以是负数，加起来也不用是任意值，只有其相对大小有意义，经过softmax才变成分数。其维度是1 x 3 x 50257。其中1是batch size，3是这三个token，50257是词表大小
    - 其中的index 0 和 1在本次并不关心，因为我们已经知道了thank后面是you和very，但是训练的时候这些内容很重要，这里也体现了一次前向传播可以并行/并列训练多个目标的性质。
+ past_key_values：KV Cache，保存了12个DynamicLayer，是GPT-2的12个Transformer Block。
+ hidden_states：不同的feature，这里默认没有保存
+ attentions: 默认没有保存，是不同的transformer block里面的attentions
+ cross_attentions: 用一个decoder关注另一个encoder的输出，GPT是self-attention模型，这一项没有

## 查看模型认为最可能的一些词

In [7]:
probabilities = torch.softmax(next_token_logits, dim=0)

top_k = 10
top_probs, top_indices = torch.topk(probabilities, top_k)

logger.info(f"模型预测 '{prompt}' 后面最可能的 {top_k} 个词：")
for i in range(top_k):
    token = tokenizer.decode(top_indices[i])
    prob = top_probs[i].item() * 100
    logger.info(f"Rank = {i + 1:2d}, token = \"{token}\", prob = {prob:5.1f}%")

2026-08-03 12:51:09.503 | INFO     | __main__:<module>:6 - 模型预测 'Thank you very' 后面最可能的 10 个词：
2026-08-03 12:51:09.503 | INFO     | __main__:<module>:10 - Rank =  1, token = " much", prob =  99.2%
2026-08-03 12:51:09.504 | INFO     | __main__:<module>:10 - Rank =  2, token = " very", prob =   0.3%
2026-08-03 12:51:09.504 | INFO     | __main__:<module>:10 - Rank =  3, token = ",", prob =   0.3%
2026-08-03 12:51:09.504 | INFO     | __main__:<module>:10 - Rank =  4, token = "much", prob =   0.1%
2026-08-03 12:51:09.504 | INFO     | __main__:<module>:10 - Rank =  5, token = " Much", prob =   0.0%
2026-08-03 12:51:09.504 | INFO     | __main__:<module>:10 - Rank =  6, token = " well", prob =   0.0%
2026-08-03 12:51:09.505 | INFO     | __main__:<module>:10 - Rank =  7, token = " highly", prob =   0.0%
2026-08-03 12:51:09.505 | INFO     | __main__:<module>:10 - Rank =  8, token = " greatly", prob =   0.0%
2026-08-03 12:51:09.505 | INFO     | __main__:<module>:10 - Rank =  9, token = " sincerel

选择其中概率最高的内容

In [8]:
best_token = tokenizer.decode(top_indices[0])
logger.info(f"选择概率最高的: \"{best_token}\"")
logger.info(f"拼接后: \"{prompt}{best_token}\"")

2026-08-03 12:51:09.516 | INFO     | __main__:<module>:2 - 选择概率最高的: " much"
2026-08-03 12:51:09.517 | INFO     | __main__:<module>:3 - 拼接后: "Thank you very much"
